# 线性回归（Linear Regression）

线性回归是机器学习中最基础的**回归**算法，用于预测连续数值型目标变量。

核心思想：找到一条直线（或超平面），使所有样本点到该直线的**残差平方和最小**。

## 1. 数学模型

**一元线性回归（Simple Linear Regression）**：

$$\hat{y} = wx + b$$

**多元线性回归（Multiple Linear Regression）**：

$$\hat{y} = w_1 x_1 + w_2 x_2 + \cdots + w_n x_n + b = \mathbf{w}^T \mathbf{x} + b$$

- $\mathbf{w} \in \mathbb{R}^n$：权重向量（控制各特征的影响力）
- $b \in \mathbb{R}$：偏置（截距）
- 目标：通过训练数据找到最优 $\mathbf{w}$ 和 $b$，使预测值 $\hat{y}$ 尽可能接近真实值 $y$

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# 生成一元线性回归示例数据（真实关系：y = 2x + 1 + 噪声）
np.random.seed(42)
x = np.linspace(0, 10, 50)
y_true = 2 * x + 1
y = y_true + np.random.randn(50) * 2

plt.figure(figsize=(8, 4))
plt.scatter(x, y, color='steelblue', s=30, alpha=0.7, label='观测数据（含噪声）')
plt.plot(x, y_true, color='red', linestyle='--', linewidth=2, label='真实直线 y = 2x + 1')
plt.xlabel('x')
plt.ylabel('y')
plt.title('一元线性回归：数据分布')
plt.legend()
plt.tight_layout()
plt.show()

## 2. 损失函数：均方误差（MSE）

**残差（Residual）**：预测值与真实值之差 $e_i = \hat{y}_i - y_i$

**均方误差（Mean Squared Error）**：

$$J(w, b) = \frac{1}{m} \sum_{i=1}^{m} (\hat{y}_i - y_i)^2$$

MSE 是关于参数 $(w, b)$ 的**二次凸函数**，存在唯一全局最小值，使求解过程稳定可靠。

| 指标 | 公式 | 特点 |
|------|------|------|
| **MSE** | $\frac{1}{m}\sum(\hat{y}-y)^2$ | 对大误差惩罚重，可微 |
| **MAE** | $\frac{1}{m}\sum|\hat{y}-y|$ | 对异常值鲁棒，不可微 |
| **Huber** | 结合 MSE 和 MAE | 兼顾两者优点 |

In [ ]:
# 演示不同权重 w 下的 MSE（固定 b=1）
w_range = np.linspace(0, 4, 300)
mse_vals = [np.mean((w * x + 1 - y) ** 2) for w in w_range]

best_w = w_range[np.argmin(mse_vals)]

plt.figure(figsize=(8, 4))
plt.plot(w_range, mse_vals, color='steelblue', linewidth=2, label='MSE(w)')
plt.axvline(best_w, color='red', linestyle='--', linewidth=1.5,
            label=f'MSE 最小处 w ≈ {best_w:.3f}')
plt.xlabel('权重 w（b 固定为 1）')
plt.ylabel('MSE')
plt.title('MSE 随权重 w 的变化（凸函数，存在全局最小值）')
plt.legend()
plt.tight_layout()
plt.show()

## 3. 解析解：正规方程

对 $J(\mathbf{w})$ 关于 $\mathbf{w}$ 求导并令其为零，可得**闭式解（正规方程）**：

$$\mathbf{w}^* = (\mathbf{X}^T \mathbf{X})^{-1} \mathbf{X}^T \mathbf{y}$$

其中 $\mathbf{X}$ 是增广特征矩阵（首列全为 1，用于吸收偏置 $b$）：

$$\mathbf{X} = \begin{bmatrix} 1 & x_1^{(1)} & \cdots & x_n^{(1)} \\ \vdots & \vdots & & \vdots \\ 1 & x_1^{(m)} & \cdots & x_n^{(m)} \end{bmatrix}$$

**优点**：精确一步求解。**缺点**：需要计算 $(\mathbf{X}^T \mathbf{X})^{-1}$，时间复杂度 $O(n^3)$，特征维度大时不适用。

In [ ]:
# 增广矩阵：第一列为 1（偏置项），第二列为特征 x
X_aug  = np.c_[np.ones(len(x)), x]          # shape: (50, 2)
w_ne   = np.linalg.inv(X_aug.T @ X_aug) @ X_aug.T @ y  # [b, w]
b_ne, w_ne_val = w_ne[0], w_ne[1]

print(f"正规方程解：w = {w_ne_val:.4f},  b = {b_ne:.4f}")
print(f"真实参数：  w = 2.0000,  b = 1.0000")

y_pred_ne = X_aug @ w_ne

plt.figure(figsize=(8, 4))
plt.scatter(x, y, color='steelblue', s=30, alpha=0.7, label='观测数据')
plt.plot(x, y_pred_ne, color='tomato', linewidth=2,
         label=f'正规方程拟合  ŷ = {w_ne_val:.2f}x + {b_ne:.2f}')
for xi, yi, yi_hat in zip(x[::5], y[::5], y_pred_ne[::5]):
    plt.plot([xi, xi], [yi, yi_hat], color='gray', alpha=0.5, linewidth=1)
plt.xlabel('x')
plt.ylabel('y')
plt.title('正规方程求解线性回归（灰线为残差）')
plt.legend()
plt.tight_layout()
plt.show()

## 4. 梯度下降求解

对损失 $J(w, b)$ 求偏导：

$$\frac{\partial J}{\partial w} = \frac{2}{m} \sum_{i=1}^{m} (\hat{y}_i - y_i)\, x_i = \frac{2}{m} X^T (\hat{y} - y)$$

$$\frac{\partial J}{\partial b} = \frac{2}{m} \sum_{i=1}^{m} (\hat{y}_i - y_i)$$

参数更新规则：

$$w \leftarrow w - \alpha \frac{\partial J}{\partial w}, \quad b \leftarrow b - \alpha \frac{\partial J}{\partial b}$$

- $\alpha$：学习率，控制每步更新幅度
- 特征尺度差异大时需**标准化**，否则梯度方向偏斜，收敛慢

In [ ]:
w_gd, b_gd = 0.0, 0.0
alpha = 0.01
losses_gd = []

for _ in range(1000):
    y_hat = w_gd * x + b_gd
    loss  = np.mean((y_hat - y) ** 2)
    losses_gd.append(loss)
    dw = 2 * np.mean((y_hat - y) * x)
    db = 2 * np.mean(y_hat - y)
    w_gd -= alpha * dw
    b_gd -= alpha * db

print(f"梯度下降解（1000步）：w = {w_gd:.4f},  b = {b_gd:.4f}")
print(f"正规方程解           ：w = {w_ne_val:.4f},  b = {b_ne:.4f}")

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].plot(losses_gd, color='steelblue')
axes[0].set_xlabel('迭代次数')
axes[0].set_ylabel('MSE')
axes[0].set_title('梯度下降收敛曲线')

axes[1].scatter(x, y, color='steelblue', s=30, alpha=0.7, label='观测数据')
axes[1].plot(x, w_gd * x + b_gd, color='tomato', linewidth=2,
             label=f'梯度下降拟合  ŷ = {w_gd:.2f}x + {b_gd:.2f}')
axes[1].set_xlabel('x')
axes[1].set_ylabel('y')
axes[1].set_title('梯度下降拟合结果')
axes[1].legend()

plt.tight_layout()
plt.show()

## 5. sklearn 实现

使用**广告投入数据集**（TV / Radio / Newspaper 广告费 → 销量预测）进行多元线性回归实战。

In [ ]:
import pandas as pd
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

df = pd.read_csv('./data/advertising.csv', index_col=0)
print(df.head())
print(f"\n数据集大小: {df.shape}")

X_ad = df.drop('Sales', axis=1).values
y_ad = df['Sales'].values
feat_names = df.drop('Sales', axis=1).columns.tolist()

X_tr, X_te, y_tr, y_te = train_test_split(X_ad, y_ad, test_size=0.2, random_state=42)

model = LinearRegression()
model.fit(X_tr, y_tr)

print(f"\n截距 b = {model.intercept_:.4f}")
for name, coef in zip(feat_names, model.coef_):
    print(f"  {name:10s}  w = {coef:.4f}")

## 6. 评估指标

| 指标 | 公式 | 含义 |
|------|------|------|
| **MSE** | $\frac{1}{m}\sum(\hat{y}-y)^2$ | 均方误差，对大误差敏感 |
| **RMSE** | $\sqrt{\text{MSE}}$ | 与 $y$ 量纲相同，直观可读 |
| **MAE** | $\frac{1}{m}\sum|\hat{y}-y|$ | 平均绝对误差，对异常值鲁棒 |
| **R²** | $1 - \dfrac{\sum(\hat{y}-y)^2}{\sum(\bar{y}-y)^2}$ | 决定系数，越接近 1 越好 |

**R² 的直觉**：模型比「只预测均值」好了多少比例。
- $R^2 = 1$：完美拟合
- $R^2 = 0$：与直接预测均值相当
- $R^2 < 0$：比均值预测还差（严重过拟合或欠拟合）

In [ ]:
y_pred_te = model.predict(X_te)

mse  = mean_squared_error(y_te, y_pred_te)
rmse = np.sqrt(mse)
mae  = mean_absolute_error(y_te, y_pred_te)
r2   = r2_score(y_te, y_pred_te)

print(f"MSE  = {mse:.4f}")
print(f"RMSE = {rmse:.4f}")
print(f"MAE  = {mae:.4f}")
print(f"R²   = {r2:.4f}")

plt.figure(figsize=(6, 5))
plt.scatter(y_te, y_pred_te, alpha=0.7, color='steelblue', edgecolors='k', s=50)
lim = [min(y_te.min(), y_pred_te.min()) - 0.5,
       max(y_te.max(), y_pred_te.max()) + 0.5]
plt.plot(lim, lim, 'r--', linewidth=2, label='完美预测线')
plt.xlabel('真实销量')
plt.ylabel('预测销量')
plt.title(f'预测值 vs 真实值（R² = {r2:.4f}）')
plt.legend()
plt.tight_layout()
plt.show()

## 7. 多元线性回归：系数解读

In [ ]:
coefs = model.coef_

colors_bar = ['steelblue' if c > 0 else 'tomato' for c in coefs]
plt.figure(figsize=(7, 4))
bars = plt.bar(feat_names, coefs, color=colors_bar, edgecolor='k', alpha=0.8)
plt.axhline(0, color='black', linewidth=0.8)
for bar, c in zip(bars, coefs):
    offset = 0.003 if c > 0 else -0.003
    va = 'bottom' if c > 0 else 'top'
    plt.text(bar.get_x() + bar.get_width() / 2, c + offset,
             f'{c:.4f}', ha='center', va=va, fontsize=10)
plt.ylabel('回归系数 w')
plt.title('各广告渠道对销量的影响（回归系数）')
plt.tight_layout()
plt.show()

print(f"TV 系数 {coefs[0]:.4f}：每增加 1 单位 TV 广告，销量平均增加 {coefs[0]:.4f} 单位")
print(f"Newspaper 系数 {coefs[2]:.4f}：与销量相关性弱（与皮尔逊相关系数分析一致）")

## 8. 多项式回归

线性回归只能拟合直线关系。通过 **PolynomialFeatures** 将原始特征扩展为多项式特征，再套线性回归，即可拟合**非线性**关系：

$$x \xrightarrow{\text{PolynomialFeatures}} [x, x^2, \ldots, x^d] \xrightarrow{\text{LinearRegression}} \hat{y}$$

本质仍是**线性模型**（关于扩展后的特征线性），但对原始输入来说是非线性的。

- degree 太低 → **欠拟合**，无法捕捉真实规律
- degree 太高 → **过拟合**，训练集误差小但泛化差

In [ ]:
from sklearn.preprocessing import PolynomialFeatures
from sklearn.pipeline import make_pipeline

np.random.seed(42)
x_p = np.sort(np.random.uniform(0, 2 * np.pi, 60))
y_p = np.sin(x_p) + np.random.randn(60) * 0.3

x_plot = np.linspace(0, 2 * np.pi, 300)

plt.figure(figsize=(10, 4))
plt.scatter(x_p, y_p, color='steelblue', s=20, alpha=0.6, label='训练数据')
plt.plot(x_plot, np.sin(x_plot), 'k--', linewidth=1.5, label='真实曲线 sin(x)', alpha=0.6)

for degree, color in zip([1, 3, 9], ['tomato', 'green', 'purple']):
    pipe = make_pipeline(PolynomialFeatures(degree), LinearRegression())
    pipe.fit(x_p.reshape(-1, 1), y_p)
    y_plot_p = pipe.predict(x_plot.reshape(-1, 1))
    r2_p = r2_score(y_p, pipe.predict(x_p.reshape(-1, 1)))
    plt.plot(x_plot, y_plot_p, color=color, linewidth=2,
             label=f'degree={degree}  (训练R²={r2_p:.3f})')

plt.ylim(-2, 2)
plt.xlabel('x')
plt.ylabel('y')
plt.title('多项式回归：欠拟合 (d=1) vs 合适 (d=3) vs 过拟合 (d=9)')
plt.legend()
plt.tight_layout()
plt.show()

## 9. 正则化：Ridge / Lasso / ElasticNet

高次多项式特征带来**过拟合**风险，正则化在损失函数中加入惩罚项来约束权重：

| 方法 | 损失函数 | 效果 |
|------|----------|------|
| **OLS**（无正则化）| $J$ | 容易过拟合 |
| **Ridge**（L2）| $J + \alpha \|\mathbf{w}\|_2^2$ | 权重整体缩小，保留所有特征 |
| **Lasso**（L1）| $J + \alpha \|\mathbf{w}\|_1$ | 部分权重归零，自动特征选择 |
| **ElasticNet** | $J + \alpha_1\|\mathbf{w}\|_1 + \alpha_2\|\mathbf{w}\|_2^2$ | L1 + L2 组合 |

`alpha` 越大，正则化越强，模型越简单（偏差↑ 方差↓）。

In [ ]:
from sklearn.linear_model import Ridge, Lasso, ElasticNet
from sklearn.preprocessing import StandardScaler

np.random.seed(0)
x_r = np.sort(np.random.uniform(0, 5, 40))
y_r = np.sin(x_r) + np.random.randn(40) * 0.3

poly10 = PolynomialFeatures(degree=10, include_bias=False)
sc_r   = StandardScaler()
X_r10  = sc_r.fit_transform(poly10.fit_transform(x_r.reshape(-1, 1)))

x_pl   = np.linspace(0, 5, 300)
X_pl10 = sc_r.transform(poly10.transform(x_pl.reshape(-1, 1)))

reg_models = {
    'OLS（无正则化）':      LinearRegression(),
    'Ridge (α=1)':         Ridge(alpha=1.0),
    'Lasso (α=0.01)':      Lasso(alpha=0.01, max_iter=10000),
    'ElasticNet (α=0.01)': ElasticNet(alpha=0.01, l1_ratio=0.5, max_iter=10000),
}

plt.figure(figsize=(11, 4))
plt.scatter(x_r, y_r, color='black', s=20, zorder=5, label='训练数据')
plt.plot(x_pl, np.sin(x_pl), 'k--', linewidth=1.5, label='真实曲线', alpha=0.5)

for (name, mdl), lc in zip(reg_models.items(), ['steelblue', 'tomato', 'green', 'purple']):
    mdl.fit(X_r10, y_r)
    plt.plot(x_pl, mdl.predict(X_pl10), color=lc, linewidth=2, label=name)

plt.ylim(-2.5, 2.5)
plt.xlabel('x')
plt.ylabel('y')
plt.title('正则化对比（10次多项式特征）')
plt.legend(fontsize=8)
plt.tight_layout()
plt.show()

lasso_fit = Lasso(alpha=0.01, max_iter=10000).fit(X_r10, y_r)
print(f"Lasso 非零权重数量：{np.sum(lasso_fit.coef_ != 0)} / {len(lasso_fit.coef_)}")

## 10. 正则化强度 α 的影响（交叉验证选参）

In [ ]:
from sklearn.model_selection import cross_val_score

alphas = np.logspace(-4, 3, 50)
ridge_cv, lasso_cv = [], []

for a in alphas:
    ridge_cv.append(-cross_val_score(
        Ridge(alpha=a), X_r10, y_r, cv=5, scoring='neg_mean_squared_error').mean())
    lasso_cv.append(-cross_val_score(
        Lasso(alpha=a, max_iter=10000), X_r10, y_r, cv=5,
        scoring='neg_mean_squared_error').mean())

best_ridge_a = alphas[np.argmin(ridge_cv)]
best_lasso_a = alphas[np.argmin(lasso_cv)]

plt.figure(figsize=(8, 4))
plt.semilogx(alphas, ridge_cv, label='Ridge CV-MSE', color='tomato',    linewidth=2)
plt.semilogx(alphas, lasso_cv, label='Lasso CV-MSE', color='steelblue', linewidth=2)
plt.axvline(best_ridge_a, color='tomato',    linestyle='--',
            label=f'Ridge 最优 α={best_ridge_a:.4f}')
plt.axvline(best_lasso_a, color='steelblue', linestyle='--',
            label=f'Lasso 最优 α={best_lasso_a:.4f}')
plt.xlabel('α（正则化强度）')
plt.ylabel('CV MSE')
plt.title('正则化强度 α vs 交叉验证 MSE')
plt.legend()
plt.tight_layout()
plt.show()

print(f"Ridge 最优 alpha = {best_ridge_a:.4f}")
print(f"Lasso 最优 alpha = {best_lasso_a:.4f}")

## 总结

| 知识点 | 核心内容 |
|--------|----------|
| 模型本质 | 预测值 = 特征的线性组合，目标最小化 MSE |
| 求解方式 | 正规方程（小数据，精确）或梯度下降（大数据，迭代） |
| 损失函数 | MSE（均方误差），二次凸函数，唯一全局最小值 |
| 评估指标 | MSE / RMSE（有量纲）、MAE（鲁棒）、R²（可解释） |
| 多项式回归 | 特征扩展 → 拟合非线性，degree 过高会过拟合 |
| 正则化 | Ridge (L2) 缩小权重；Lasso (L1) 产生稀疏权重；α 越大越保守 |

**适用场景：**
- 房价预测、销量预测、温度预测等连续值回归任务
- 优点：简单、快速、可解释性强、训练高效
- 局限：线性假设，无法捕捉复杂非线性模式（需多项式/核方法/神经网络）